In [ ]:
CONFIG = {
    "api_key": "6e6716df-b6d7-4989-9112-e1e157510423",

    # Mandatory ENTSO-E timezone
    "timezone": "Europe/Brussels",

    # Time window
    "start": "2025-01-01 00:00",
    "end":   "2025-01-02 00:00",

    # Bidding zones / countries
    "zones": ["DE_LU", "NL", "FR"],

    # Toggle datasets
    "datasets": {
        "day_ahead_prices": True,
        "intraday_prices": True,
        "load_actual": True,
        "load_forecast": True,
        "generation_actual": True,
        "wind_solar_forecast": True,
        "cross_border_flows": True,
        "balancing_prices": True,
        "imbalance_prices": True,
    }
}


: 

In [ ]:
import pandas as pd
import pytz
from entsoe import EntsoePandasClient

def fetch_entsoe_data(config):
    client = EntsoePandasClient(api_key=config["api_key"])

    tz = pytz.timezone(config["timezone"])
    start = pd.Timestamp(config["start"], tz=tz)
    end   = pd.Timestamp(config["end"], tz=tz)

    results = {}

    for zone in config["zones"]:
        zone_data = {}

        if config["datasets"].get("day_ahead_prices"):
            zone_data["day_ahead_prices"] = client.query_day_ahead_prices(
                country_code=zone, start=start, end=end
            )

        if config["datasets"].get("intraday_prices"):
            zone_data["intraday_prices"] = client.query_intraday_prices(
                country_code=zone, start=start, end=end
            )

        if config["datasets"].get("load_actual"):
            zone_data["load_actual"] = client.query_load(
                country_code=zone, start=start, end=end
            )

        if config["datasets"].get("load_forecast"):
            zone_data["load_forecast"] = client.query_load_forecast(
                country_code=zone, start=start, end=end
            )

        if config["datasets"].get("generation_actual"):
            zone_data["generation_actual"] = client.query_generation(
                country_code=zone, start=start, end=end, psr_type=None
            )

        if config["datasets"].get("wind_solar_forecast"):
            zone_data["wind_solar_forecast"] = client.query_wind_and_solar_forecast(
                country_code=zone, start=start, end=end
            )

        if config["datasets"].get("cross_border_flows"):
            zone_data["cross_border_flows"] = client.query_crossborder_flows(
                country_code_from=zone,
                country_code_to=None,
                start=start,
                end=end
            )

        if config["datasets"].get("balancing_prices"):
            zone_data["balancing_prices"] = client.query_balancing_prices(
                country_code=zone, start=start, end=end
            )

        if config["datasets"].get("imbalance_prices"):
            zone_data["imbalance_prices"] = client.query_imbalance_prices(
                country_code=zone, start=start, end=end
            )

        results[zone] = zone_data

    return results


In [ ]:
data = fetch_entsoe_data(CONFIG)

# Example: DE intraday prices
print(data["DE_LU"]["intraday_prices"].head())

# Example: NL generation mix
print(data["NL"]["generation_actual"].head())
